# Sycophancy under pushback: English vs Spanish - Fatima Challenge

Each run is saved to Google Drive under
`/content/drive/My Drive/Research/Sycophancy/results-DD-MM-YYYY-HH-MM`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Results folder on Google Drive (one new folder per run)

In [ ]:
import os
from datetime import datetime
from zoneinfo import ZoneInfo
from google.colab import drive

drive.mount("/content/drive")
TIMEZONE = "UTC" 
RUN_NAME = "results-" + datetime.now(ZoneInfo(TIMEZONE)).strftime("%d-%m-%Y-%H-%M")
RUN_DIR = f"/content/drive/My Drive/Research/Sycophancy/{RUN_NAME}"

os.makedirs(RUN_DIR, exist_ok=True)
print(RUN_DIR)

## 2. Code and dependencies

In [ ]:
!git clone -q -b claude/refactor-simple https://github.com/Amos-Luna/sycophancy-gap-research.git
%cd /content/sycophancy-gap-research
!pip install -q -r requirements.txt
!python -m pytest -q tests/

## 4. Full evaluation (3 model sizes)

In [ ]:
MODELS = [("Qwen/Qwen3.5-0.8B", 32), ("Qwen/Qwen3.5-2B", 32), ("Qwen/Qwen3.5-4B", 16)]  # (model, batch size)
for model, batch in MODELS:
    out = f"{RUN_DIR}/{model.split('/')[-1]}"
    !python src/evaluate.py --model {model} --batch-size {batch} --out "{out}" 2>&1 | grep -v 'falling back'

## 5. Analysis

In [ ]:
from IPython.display import Image, Markdown, display
!python src/analyze.py "{RUN_DIR}" > /dev/null
display(Markdown(open(f"{RUN_DIR}/summary.md").read()))
display(Image(f"{RUN_DIR}/abandonment.png"), Image(f"{RUN_DIR}/discrimination.png"))

## 6. Download the results as a zip 
format: `results-DD-MM-YYYY-HH-MM.zip`

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive(f"/content/{RUN_NAME}", "zip", root_dir=os.path.dirname(RUN_DIR), base_dir=RUN_NAME)
files.download(zip_path)

## 7. Publish on Hugging Face

In [ ]:
import getpass
os.environ["HF_TOKEN"] = getpass.getpass("HF token: ")
REPO = "amosluna/sycophancy-gap-es-en"  #My Own HF username
RESULTS = "results"
COPY_MODELS = False                  # True also mirrors the exact model weights (~15 GB)
FLAG = "--copy-models" if COPY_MODELS else ""
!python src/upload.py --repo {REPO} --results "{RESULTS}" {FLAG}